# Data preparation

1. Satellite data download
2. Generate training datacubes
3. Generate inference datacubes
4. Generate training data

In [2]:
import os
import datetime
import geopandas as gpd
import multiprocessing as mp
import pandas as pd
import numpy as np

import cdseutils.utils

import fetch_satdata.download.download_sentinel2_from_s3
import fetch_satdata.datacube.setup_datacube_run
import fetch_satdata.datacube.datacube_flatten_2d
import fetch_satdata.workflows.create_datacube

In [3]:
CDSE_JSON_FILEPATH = '/Users/nikhilsrajan/NASA-Harvest/project/fetch_satdata/data/cdse_credentials.json'

DEMO_REGION_FILEPATH = '/Users/nikhilsrajan/NASA-Harvest/project/fetch_satdata/data/at_test_download_geometry.geojson'

# Training data
TRAINING_DATA_FILEPATH = '/Users/nikhilsrajan/NASA-Harvest/project/fetch_satdata/data/at_eurocrops_sel_region_and_crops_2018_sampled.geojson'
TRAINING_ID_COL = 'fid' # column name in the TRAINING_DATA_FILEPATH which uniquely identifies each polygon
TRAINING_LABEL_COL = 'EC_hcat_n' # column name in the TRAINING_DATA_FILEPATH which tells the label corresponding to the polygon
TRAINING_DATACUBE_FOLDERPATH = '/Users/nikhilsrajan/NASA-Harvest/project/fetch_satdata/data/at_eurocrops/training_datacubes'
TRAINING_DATA_FOLDERPATH = '/Users/nikhilsrajan/NASA-Harvest/project/fetch_satdata/data/at_eurocrops/training_data'

# Inference data
INFERENCE_S2GRIDS_FILEPATH = '/Users/nikhilsrajan/NASA-Harvest/project/fetch_satdata/data/at_test_inference_s2_grids.geojson'
INFERENCE_ID_COL = 'id'
INFERENCE_DATACUBE_FOLDERPATH = '/Users/nikhilsrajan/NASA-Harvest/project/fetch_satdata/data/at_eurocrops/inference_datacubes'

START_DATE_STR = '2018-01-01'
END_DATE_STR = '2018-12-31'

SENTINEL2_LEVEL = 'l2a'
BANDS = ['B02', 'B03', 'B04', 'B05', 'B06', 'B07', 'B08', 'B11', 'B12', 'SCL']
MOSAIC_DAYS = 20
SCL_MASK_CLASSES = [0, 1, 3, 7, 8, 9, 10]

TRAINING_NJOBS = mp.cpu_count() - 2
INFERENCE_NJOBS = 4

# TO DO: hide the paths below + put the three functions into one call
TILE_DOWNLOAD_FOLDERPATH = '/Users/nikhilsrajan/NASA-Harvest/project/fetch_satdata/data/at_eurocrops/satellite/sentinel-2-l2a'

In [4]:
CDSE_CREDS = cdseutils.utils.cdse_credentials_from_json(CDSE_JSON_FILEPATH)

START_DATE = datetime.datetime.strptime(START_DATE_STR, '%Y-%m-%d')
END_DATE = datetime.datetime.strptime(END_DATE_STR, '%Y-%m-%d')

SENTINEL2_CATALOG_FILEPATH = os.path.join(TILE_DOWNLOAD_FOLDERPATH, 'catalog_sentinel-2.geojson')

TIMESTAMP_COL = 'timestamp' # column name in the SENTINEL2_CATALOG_FILEPATH which tells the datetime of image acquisition

TRAINING_DATACUBE_CATALOG_FILEPATH = os.path.join(TRAINING_DATACUBE_FOLDERPATH, 'input.csv')
INFERENCE_DATACUBE_CATALOG_FILEPATH = os.path.join(INFERENCE_DATACUBE_FOLDERPATH, 'input.csv')

## Download satellite data + create datacubes

In [ ]:
successful_download_count, total_download_count = \
fetch_satdata.download.download_sentinel2_from_s3.main(
    cdse_creds = CDSE_CREDS,
    roi_filepath = DEMO_REGION_FILEPATH,
    startdate = START_DATE,
    enddate = END_DATE,
    upper_limit_for_number_of_tiles = 147, # 241
    root_folderpath = TILE_DOWNLOAD_FOLDERPATH,
    sentinel2_local_catalog_filepath = SENTINEL2_CATALOG_FILEPATH,
    level = SENTINEL2_LEVEL,
    bands = BANDS,
    chunksize_for_download_and_update_catalog = 1617, # just for demo, otherwise keep it as 100
)

# 42m for 1617 files (147 tiles)

In [ ]:
fetch_satdata.workflows.create_datacube.run_create_datacube(
    catalog_filepath = SENTINEL2_CATALOG_FILEPATH,
    timestamp_col = TIMESTAMP_COL,
    shapefilepath = TRAINING_DATA_FILEPATH,
    id_col = TRAINING_ID_COL,
    run_folderpath = TRAINING_DATACUBE_FOLDERPATH,
    startdate = START_DATE,
    enddate = END_DATE,
    bands = BANDS,
    scl_mask_classes = SCL_MASK_CLASSES,
    mosaic_days = MOSAIC_DAYS,
    csv_filepath = TRAINING_DATACUBE_CATALOG_FILEPATH,
    label_col = TRAINING_LABEL_COL,
    cores = TRAINING_NJOBS,
)

In [ ]:
fetch_satdata.workflows.create_datacube.run_create_datacube(
    catalog_filepath = SENTINEL2_CATALOG_FILEPATH,
    timestamp_col = TIMESTAMP_COL,
    shapefilepath = INFERENCE_S2GRIDS_FILEPATH,
    id_col = INFERENCE_ID_COL,
    run_folderpath = INFERENCE_DATACUBE_FOLDERPATH,
    startdate = START_DATE,
    enddate = END_DATE,
    bands = BANDS,
    scl_mask_classes = SCL_MASK_CLASSES,
    mosaic_days = MOSAIC_DAYS,
    csv_filepath = INFERENCE_DATACUBE_CATALOG_FILEPATH,
    label_col = None,
    cores = INFERENCE_NJOBS,
    # unlock = True,
    dry_run = True
)

In [ ]:
training_datacube_filepaths_df = pd.read_csv(TRAINING_DATACUBE_CATALOG_FILEPATH)

training_datacube_filepaths_df.head()

In [ ]:
inference_datacube_filepaths_df = pd.read_csv(INFERENCE_DATACUBE_CATALOG_FILEPATH)

inference_datacube_filepaths_df.head()

## Example of datacube data

In [10]:
def get_largest_datacube(
    shape_filepath,
    id_col,
    datacubes_csv_filepath,
):
    largest_id = gpd.read_file(shape_filepath).set_index(id_col).area.idxmax()
    datacube_folderpath = pd.read_csv(datacubes_csv_filepath).set_index('id')['export_folderpath'].to_dict()[largest_id]
    datacube_filepath = os.path.join(datacube_folderpath, 'datacube.npy')
    metadata_filepath = os.path.join(datacube_folderpath, 'metadata.pickle.npy')
    datacube = np.load(datacube_filepath)
    metadata = np.load(metadata_filepath, allow_pickle=True)[()]
    return datacube, metadata

In [11]:
import rsutils.modify_bands
import matplotlib.pyplot as plt
import seaborn as sns


def interp_compute_ndvi(
    data, 
    band_indices, 
    data_type = 'datacube' # options: datacube, flattened_datacubes
):
    def _expand_data(data, data_type):
        """ 
        This function is to use "modify_bands" which expects the data to be 5d
        numpy array with dimensions (samples, timestamps, height, width, bands)
        """
        if data_type == 'datacube':
            # datacube is 4d data with dimesions (timestamps, height, width, bands)
            return np.expand_dims(data, axis=0)
        
        elif data_type == 'flattened_datacubes':
            # flattened_datacubes is a 3d data with dimensions (pixels, timestamps, bands)
            return np.expand_dims(data, axis=(2, 3))
        
        else:
            raise ValueError(f'Invalid data_type={data_type}.')



    interp_data, interp_band_indices = \
    rsutils.modify_bands.modify_bands(
        bands = _expand_data(data = data, data_type = data_type),
        band_indices = band_indices.copy(),
        sequence = [
            (rsutils.modify_bands.mask_invalid_and_interpolate, {}),
            (rsutils.modify_bands.compute_bands, dict(bands_to_compute = ['NDVI'])),
        ]
    )

    interp_data = np.squeeze(interp_data)

    return interp_data, interp_band_indices


def plot_timeseries(
    ndvi_datacube,
    timestamps,
    title_suffix = '',
    plot_n_timeseries = 100,
):
    ts, h, w = ndvi_datacube.shape
    y_indices, x_indices = np.where(~np.isnan(ndvi_datacube).any(axis=0))
    ndvi_datacube_2d = ndvi_datacube[:, y_indices, x_indices]

    scale = 5
    aspect_ratio = 1.5
    fig, ax = plt.subplots(figsize=(scale*aspect_ratio, scale))

    for ts in ndvi_datacube_2d.T[:plot_n_timeseries]:
        ax.plot(timestamps, ts, 
                color='green', linewidth=1, alpha=0.2)
        

    ax.plot(timestamps, np.median(ndvi_datacube_2d, axis=1), 
            color='black', linewidth=1, linestyle='--', alpha=1, label='Median')

    ax.legend()

    ax.set_title(f'First {plot_n_timeseries} NDVI timeseries' + title_suffix)

    return fig


def plot_2d(data_2d, title=None):
    h, w = data_2d.shape

    scale = 5
    aspect_ratio = w / h
    fig, ax = plt.subplots(figsize=(scale*aspect_ratio, scale))

    g = sns.heatmap(
        ax = ax, 
        data = data_2d,
        vmin = 0.2,
        vmax = 0.8,
    )

    if title is not None:
        g.set_title(title)


def plot_ndvi_data(datacube, metadata):
    bands = metadata.get('bands')
    band_indices = {band: index for index, band in enumerate(bands)}

    interp_datacube, interp_band_indices = interp_compute_ndvi(
        data = datacube, band_indices = band_indices, data_type = 'datacube'
    )

    ndvi_datacube = interp_datacube[:,:,:,interp_band_indices['NDVI']]

    plot_timeseries(
        ndvi_datacube = ndvi_datacube,
        timestamps = metadata.get('timestamps'),
    )

    plot_2d(data_2d = ndvi_datacube.max(axis=0), title = 'Max NDVI')


In [ ]:
l_train_datacube, l_train_metadata = get_largest_datacube(
    shape_filepath = TRAINING_DATA_FILEPATH, 
    id_col = TRAINING_ID_COL, 
    datacubes_csv_filepath = TRAINING_DATACUBE_CATALOG_FILEPATH,
)

plot_ndvi_data(datacube = l_train_datacube, metadata = l_train_metadata)

In [ ]:
l_inf_datacube, l_inf_metadata = get_largest_datacube(
    shape_filepath = INFERENCE_S2GRIDS_FILEPATH,
    id_col = INFERENCE_ID_COL, 
    datacubes_csv_filepath = INFERENCE_DATACUBE_CATALOG_FILEPATH,
)

plot_ndvi_data(datacube = l_inf_datacube, metadata = l_inf_metadata)

## Aggregating datacubes into training data

In [ ]:
fetch_satdata.datacube.datacube_flatten_2d.main(
    filepaths_df = pd.read_csv(TRAINING_DATACUBE_CATALOG_FILEPATH),
    filepath_col = 'datacube_filepath',
    id_col = 'id',
    label_col = 'label',
    export_folderpath = TRAINING_DATA_FOLDERPATH,
)

## Plotting training data

In [15]:
training_data = np.load(os.path.join(TRAINING_DATA_FOLDERPATH, 'data.npy'))
ids = np.load(os.path.join(TRAINING_DATA_FOLDERPATH, 'ids.npy'))
labels = np.load(os.path.join(TRAINING_DATA_FOLDERPATH, 'labels.npy'))
training_metadata = np.load(os.path.join(TRAINING_DATA_FOLDERPATH, 'metadata.pickle.npy'), allow_pickle=True)[()]

In [16]:
def median_per_id(ids, data, labels):
    _, n_ts, n_bands = data.shape
    
    unique_ids, inverse = np.unique(ids, return_inverse=True)
    n_ids = len(unique_ids)

    data_median = np.zeros(shape=(n_ids, n_ts, n_bands), dtype=data.dtype)
    labels_median = np.zeros(n_ids, dtype=labels.dtype)
    
    for i in range(n_ids):
        data_median[i] = np.nanmedian(data[inverse == i], axis=0)
        labels_median[i] = labels[inverse == i][0]

    ids_median = unique_ids

    return ids_median, data_median, labels_median    

In [17]:
def plot_training_data(
    training_data,
    training_metadata,
    ids,
    labels,
):
    bands = training_metadata.get('bands')
    band_indices = {band: index for index, band in enumerate(bands)}

    interp_training_data, interp_band_indices = \
    interp_compute_ndvi(
        data = training_data,
        band_indices = band_indices,
        data_type = 'flattened_datacubes',
    )

    ids_median, interp_training_data_median, labels_median = \
    median_per_id(
        ids = ids,
        data = interp_training_data,
        labels = labels,
    )

    interp_training_ndvi_median = interp_training_data_median[:,:,interp_band_indices['NDVI']]

    n_ids, n_timestamps = interp_training_ndvi_median.shape

    df = pd.DataFrame({
        "id": np.repeat(ids_median, n_timestamps),
        "label": np.repeat(labels_median, n_timestamps),
        "timestamp": np.tile(training_metadata['timestamps'], n_ids),
        "NDVI": interp_training_ndvi_median.reshape(-1)
    })

    scale = 5
    aspect_ratio = 2
    fig, ax = plt.subplots(figsize=(scale*aspect_ratio, scale))

    g = sns.lineplot(
        data = df,
        x = 'timestamp',
        y = 'NDVI',
        hue = 'label',
        estimator = 'median',
    )

    g.legend(title="label", bbox_to_anchor=(1.05, 1), loc="upper left")

In [ ]:
plot_training_data(
    training_data = training_data,
    training_metadata = training_metadata,
    ids = ids,
    labels = labels,
)